In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
file_path = '/content/drive/MyDrive/Res/final_processed.csv'
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does not exist.")

The file '/content/drive/MyDrive/Res/final_processed.csv' exists.


In [ ]:
import pandas as pd
import torch

# Verify GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Change runtime before proceeding.")

# Verify dataset
df = pd.read_csv(file_path)
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample row:")
print(df['content'].iloc[0][:200])

GPU available: True
Device: Tesla T4

Dataset shape: (72000, 2)
Columns: ['content', 'label']
Label distribution:
label
1    37030
0    34970
Name: count, dtype: int64

Sample row:
law enforcement high alert following threat cop white 911by blacklivesmatter fyf911 terrorist video comment expected barack obama member fyf911 fukyoflag blacklivesmatter movement called lynching hang


In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_token'))
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [ ]:
df = df[['content','label']].dropna()
df = df[df['content'].str.strip().str.len() > 10]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42,
    stratify=train_df['label']
)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 51814 | Val: 5758 | Test: 14393


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
  return tokenizer(
        batch['content'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))
train_dataset = train_dataset.map(tokenize, batched=True, batch_size=64)
val_dataset   = val_dataset.map(tokenize, batched=True, batch_size=64)
test_dataset  = test_dataset.map(tokenize, batched=True, batch_size=64)

cols = ['input_ids', 'attention_mask', 'label']
train_dataset.set_format(type='torch', columns=cols)
val_dataset.set_format(type='torch', columns=cols)
test_dataset.set_format(type='torch', columns=cols)

print("Tokenization complete.")
print(f"Sample shape: {train_dataset[0]['input_ids'].shape}")

Map:   0%|          | 0/51814 [00:00<?, ? examples/s]

Map:   0%|          | 0/5758 [00:00<?, ? examples/s]

Map:   0%|          | 0/14393 [00:00<?, ? examples/s]

Tokenization complete.
Sample shape: torch.Size([512])


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Real", 1: "Fake"},
    label2id={"Real": 0, "Fake": 1}
)
print("Model loaded.")
print(f"Parameters: {model.num_parameters():,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Parameters: 66,955,010


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='distilbert_fakenews',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.022987,0.053586,0.987843,0.987845
2,0.025779,0.030534,0.991316,0.991317
3,0.000141,0.033311,0.993574,0.993574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=9717, training_loss=0.04534574214797748, metrics={'train_runtime': 2087.8297, 'train_samples_per_second': 74.451, 'train_steps_per_second': 4.654, 'total_flos': 2.059099738188595e+16, 'train_loss': 0.04534574214797748, 'epoch': 3.0})

In [ ]:
# Cell — Final evaluation on test set
results = trainer.evaluate(test_dataset)
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"Test F1:       {results['eval_f1']:.4f}")

Test Accuracy: 0.9927
Test F1:       0.9927


In [ ]:
# Cell — Save model
trainer.save_model('distilbert_finetuned')
tokenizer.save_pretrained('distilbert_finetuned')
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [ ]:
# Cell — Download
!zip -r distilbert_finetuned.zip distilbert_finetuned/
from google.colab import files
files.download('distilbert_finetuned.zip')

  adding: distilbert_finetuned/ (stored 0%)
  adding: distilbert_finetuned/training_args.bin (deflated 53%)
  adding: distilbert_finetuned/model.safetensors (deflated 8%)
  adding: distilbert_finetuned/config.json (deflated 51%)
  adding: distilbert_finetuned/tokenizer.json (deflated 71%)
  adding: distilbert_finetuned/tokenizer_config.json (deflated 42%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>